# 05_01 — Diagnóstico y benchmarking de Spark

**Proyecto:** Big Data con PySpark + Docker  
**Fase:** 05 — Optimización y rendimiento  
**Datos:** NYC Yellow Taxi Trip Records, 2024  
**Objetivo:** establecer una **línea base reproducible** que permita evaluar mejoras de rendimiento en los próximos notebooks.

Este notebook **no busca optimizar todavía**: primero medimos cómo se comporta Spark con configuraciones y operaciones definidas. El período es **configurable**: se han registrado la línea base de enero y la del **año completo de 2024**. Las consultas medidas y las salidas ejecutadas se conservan sin modificaciones.

**Actualización:** se añadió una guía para acceder a la **Spark UI desde Docker/VS Code** y las **respuestas desarrolladas** a las preguntas de interpretación, utilizando los resultados registrados.

> **Ejecución:** abrir el notebook con el kernel Python del contenedor Docker, como en las fases anteriores. Los archivos Parquet deben estar accesibles desde ese contenedor.


## 1. Preguntas que queremos responder

1. ¿Qué configuración de Spark y qué tamaño de entrada estamos usando?
2. ¿Qué operaciones generan `Exchange`, `HashAggregate`, `Sort` o planes adaptativos?
3. ¿Cuánto tarda un mismo *workload* bajo condiciones repetibles?
4. ¿Qué variabilidad existe entre ejecuciones y cuál será nuestra referencia para comparar optimizaciones?

### Convenciones

- **Workload**: consulta o transformación específica que Spark debe ejecutar.
- **Acción**: operación, como `collect()`, que obliga a materializar el cálculo diferido (*lazy evaluation*).
- **Shuffle**: redistribución de datos entre particiones, frecuente en agregaciones y joins.
- **AQE**: *Adaptive Query Execution*, que permite ajustar partes del plan en tiempo de ejecución.
- **Benchmark**: medición repetida de rendimiento en un entorno y condiciones documentados.

**Criterio de comparación:** no declararemos que una optimización es mejor por una única ejecución. Compararemos las **medianas** de varias repeticiones, con el mismo conjunto de datos, consulta y entorno.

## 2. Importaciones y detección del repositorio

El notebook usa la biblioteca estándar y PySpark. No necesitamos `pandas`, `matplotlib` ni paquetes adicionales para ejecutar los benchmarks.

**Importante:** las rutas se resuelven dentro del contenedor. Por ejemplo, una ruta de Windows no funcionará en un kernel que corre en Linux salvo que esté montada allí.

In [1]:
from __future__ import annotations

import json
import os
import platform
import statistics
import sys
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Callable

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


def encontrar_raiz_proyecto(punto_inicial: Path) -> Path:
    """Encuentra la carpeta del repositorio por su estructura característica."""
    punto_inicial = punto_inicial.resolve()
    for carpeta in (punto_inicial, *punto_inicial.parents):
        if (carpeta / "data").is_dir() and (carpeta / "README.md").is_file():
            return carpeta
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto (carpetas data/ y README.md). "
        "Abre el notebook desde el repositorio montado en Docker."
    )


PROJECT_ROOT = encontrar_raiz_proyecto(Path.cwd())
DATA_ROOT = PROJECT_ROOT / "data"

print("Proyecto encontrado:", PROJECT_ROOT)
print("Directorio de datos:", DATA_ROOT)
print("Sistema operativo:", platform.system())
print("Ejecutable Python:", sys.executable)

Proyecto encontrado: /workspace
Directorio de datos: /workspace/data
Sistema operativo: Linux
Ejecutable Python: /usr/local/bin/python3.11


## 3. Iniciar Spark y registrar el entorno

Para que el benchmark sea comparable, **no modificamos aún** `spark.sql.shuffle.partitions`, AQE ni otras configuraciones. Registramos sus valores reales, que deberán mantenerse constantes en una comparación justa.

Si ya existe una sesión, `getOrCreate()` la reutilizará. Para experimentar posteriormente con configuraciones estáticas, conviene reiniciar el kernel/contenedor.

In [2]:
spark = (
    SparkSession.builder
    .appName("05_01_diagnostico_benchmarking")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


def configuracion_spark() -> dict:
    """Recupera metadatos útiles del entorno de ejecución."""
    sc = spark.sparkContext
    sc_conf = sc.getConf()

    return {
        "spark_version": spark.version,
        "spark_app_id": sc.applicationId,
        "spark_app_name": sc.appName,
        "spark_master": sc.master,
        "default_parallelism": sc.defaultParallelism,
        "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
        "adaptive_query_execution": spark.conf.get("spark.sql.adaptive.enabled"),
        "adaptive_coalesce_partitions": spark.conf.get(
            "spark.sql.adaptive.coalescePartitions.enabled"
        ),
        "session_time_zone": spark.conf.get("spark.sql.session.timeZone"),
        "driver_memory": sc_conf.get("spark.driver.memory", "no especificada"),
        "executor_memory": sc_conf.get("spark.executor.memory", "no especificada"),
        "python_version": platform.python_version(),
        "system": platform.system(),
        "logical_cpu_count": os.cpu_count(),
        "spark_ui_url": sc.uiWebUrl,
    }


ENTORNO = configuracion_spark()
for clave, valor in ENTORNO.items():
    print(f"{clave:30s}: {valor}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/15 23:31:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/15 23:31:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/15 23:31:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


spark_version                 : 4.1.3
spark_app_id                  : local-1792107070606
spark_app_name                : 05_01_diagnostico_benchmarking
spark_master                  : local[*]
default_parallelism           : 12
shuffle_partitions            : 200
adaptive_query_execution      : true
adaptive_coalesce_partitions  : true
session_time_zone             : Etc/UTC
driver_memory                 : no especificada
executor_memory               : no especificada
python_version                : 3.11.17
system                        : Linux
logical_cpu_count             : 12
spark_ui_url                  : http://455f3b1cac15:4042


## 4. Selección del período y localización del Parquet

El experimento principal utiliza **los 12 meses de 2024**. Si quieres repetir la línea base de enero, cambia `MESES` por `['2024-01']` antes de ejecutar la celda siguiente.

La selección se realiza por archivos de nombre `yellow_tripdata_YYYY-MM.parquet` dentro de `data/`. Si se encuentran copias duplicadas, se exige una ruta explícita en lugar de elegir una arbitrariamente.

> **Control experimental:** conserva el mismo período, los mismos archivos, los mismos filtros y las mismas consultas para comparar una versión *baseline* y una versión optimizada. Enero vs. año completo es una comparación de **escala**, no de optimización.

In [3]:
MESES = [f"2024-{mes:02d}" for mes in range(1, 13)]  # Modifica este parámetro para ampliar el período.

# Si tienes archivos duplicados, especifica la ruta exacta del mes afectado.
# Ejemplo: RUTAS_EXPLICITAS = {"2024-01": "/workspace/data/raw/taxi/yellow_tripdata_2024-01.parquet"}
RUTAS_EXPLICITAS: dict[str, str] = {}


def siguiente_mes(fecha: datetime) -> datetime:
    """Devuelve el primer día del mes siguiente."""
    return (fecha.replace(day=28) + timedelta(days=4)).replace(day=1)


def localizar_parquet(mes: str) -> Path:
    """Resuelve un archivo mensual; evita seleccionar copias ambiguas."""
    if mes in RUTAS_EXPLICITAS:
        ruta = Path(RUTAS_EXPLICITAS[mes]).expanduser().resolve()
        if not ruta.exists():
            raise FileNotFoundError(f"No existe la ruta indicada para {mes}: {ruta}")
        return ruta

    coincidencias = sorted(DATA_ROOT.rglob(f"yellow_tripdata_{mes}.parquet"))
    if not coincidencias:
        raise FileNotFoundError(
            f"No se encontró yellow_tripdata_{mes}.parquet bajo {DATA_ROOT}. "
            "Verifica que data/ esté montada en el contenedor."
        )
    if len(coincidencias) > 1:
        opciones = "\n".join(f"  - {ruta}" for ruta in coincidencias)
        raise RuntimeError(
            f"Se encontraron {len(coincidencias)} copias del mes {mes}:\n{opciones}\n"
            "Selecciona la correcta mediante RUTAS_EXPLICITAS."
        )
    return coincidencias[0]


# Mantener meses ordenados y sin duplicados facilita reproducibilidad.
MESES = sorted(set(MESES))
assert MESES and all(m.startswith("2024-") and len(m) == 7 for m in MESES)

ARCHIVOS = [localizar_parquet(mes) for mes in MESES]
FECHA_INICIO = datetime.strptime(MESES[0], "%Y-%m")
FECHA_FIN_EXCLUSIVA = siguiente_mes(datetime.strptime(MESES[-1], "%Y-%m"))


def ruta_publicable(ruta: Path) -> str:
    """Evita incluir rutas absolutas del equipo en el reporte."""
    try:
        return str(ruta.relative_to(PROJECT_ROOT))
    except ValueError:
        return ruta.name


ARCHIVOS_RELATIVOS = [ruta_publicable(ruta) for ruta in ARCHIVOS]
TAMANO_ENTRADA_MB = sum(
    ruta.stat().st_size for ruta in ARCHIVOS if ruta.is_file()
) / (1024 ** 2)

print("Meses seleccionados:", MESES)
print("Intervalo:", FECHA_INICIO, "a", FECHA_FIN_EXCLUSIVA, "(exclusivo)")
print("Archivos:", *ARCHIVOS_RELATIVOS, sep="\n - ")
print(f"Tamaño físico (archivos individuales): {TAMANO_ENTRADA_MB:,.2f} MiB")

Meses seleccionados: ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']
Intervalo: 2024-01-01 00:00:00 a 2025-01-01 00:00:00 (exclusivo)
Archivos:
 - data/raw/taxi/year=2024/yellow_tripdata_2024-01.parquet
 - data/raw/taxi/year=2024/yellow_tripdata_2024-02.parquet
 - data/raw/taxi/year=2024/yellow_tripdata_2024-03.parquet
 - data/raw/taxi/year=2024/yellow_tripdata_2024-04.parquet
 - data/raw/taxi/year=2024/yellow_tripdata_2024-05.parquet
 - data/raw/taxi/year=2024/yellow_tripdata_2024-06.parquet
 - data/raw/taxi/year=2024/yellow_tripdata_2024-07.parquet
 - data/raw/taxi/year=2024/yellow_tripdata_2024-08.parquet
 - data/raw/taxi/year=2024/yellow_tripdata_2024-09.parquet
 - data/raw/taxi/year=2024/yellow_tripdata_2024-10.parquet
 - data/raw/taxi/year=2024/yellow_tripdata_2024-11.parquet
 - data/raw/taxi/year=2024/yellow_tripdata_2024-12.parquet
Tamaño físico (archivos individuales): 660.90 MiB


## 5. Comprobar el esquema sin calcular todavía el benchmark

La lectura de Parquet utiliza **evaluación perezosa**: definir un DataFrame no equivale a procesar todas sus filas. La siguiente verificación carga metadatos y muestra un ejemplo breve.

No usamos `count()` como principal medida de rendimiento: en ciertas condiciones Spark puede aprovechar metadatos y evitar un escaneo representativo de nuestro ETL.

In [4]:
COLUMNAS_REQUERIDAS = {
    "tpep_pickup_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "total_amount",
}

# Usamos la misma lectura en todo el experimento.
def leer_taxi() -> DataFrame:
    return spark.read.parquet(*(str(ruta) for ruta in ARCHIVOS))


muestra = leer_taxi()
columnas_faltantes = sorted(COLUMNAS_REQUERIDAS.difference(muestra.columns))
if columnas_faltantes:
    raise ValueError(f"El Parquet no tiene las columnas requeridas: {columnas_faltantes}")

muestra.printSchema()
muestra.select(*sorted(COLUMNAS_REQUERIDAS)).limit(3).show(truncate=False)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+------------+------------+------------+--------------------+-------------+
|DOLocationID|PULocationID|total_amount|tpep_pick

## 6. Definición de los workloads

Para establecer una referencia útil, elegimos operaciones de transformación comunes en un ETL.

### Workload A — Agregación por zona de origen

1. Leer columnas necesarias de viajes.
2. Filtrar fecha, distancias no positivas o extremadamente grandes y montos negativos.
3. Agrupar por `PULocationID`.
4. Calcular número de viajes, distancia media e importe medio.

### Workload B — Rutas más frecuentes

1. Aplicar los **mismos filtros** sobre el mismo período.
2. Agrupar por par `(PULocationID, DOLocationID)`.
3. Obtener cantidad de viajes e ingreso acumulado.
4. Ordenar por frecuencia y seleccionar las 20 rutas principales.

Los filtros son **reglas de este experimento**, no una definición oficial de viajes válidos. Se conservarán iguales en todos los ensayos de comparación.

**¿Por qué estas consultas?** Ambas implican lectura columnar, filtrado y agregación; la segunda añade agrupación de mayor cardinalidad y ordenamiento. Son cargas apropiadas para observar planes físicos, shuffle y tiempos.

In [5]:
def viajes_filtrados() -> DataFrame:
    """Construye la misma entrada lógica para los dos workloads."""
    df = leer_taxi().select(
        "tpep_pickup_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "total_amount",
    )

    return df.filter(
        (F.col("tpep_pickup_datetime") >= F.lit(FECHA_INICIO))
        & (F.col("tpep_pickup_datetime") < F.lit(FECHA_FIN_EXCLUSIVA))
        & F.col("PULocationID").isNotNull()
        & F.col("DOLocationID").isNotNull()
        & (F.col("trip_distance") > 0)
        & (F.col("trip_distance") < 100)
        & (F.col("total_amount") >= 0)
    )


def consulta_zonas() -> DataFrame:
    """Workload A: indicadores agregados por zona de origen."""
    return (
        viajes_filtrados()
        .groupBy("PULocationID")
        .agg(
            F.count("*").alias("num_viajes"),
            F.avg("trip_distance").alias("distancia_media"),
            F.avg("total_amount").alias("importe_medio"),
        )
    )


def consulta_rutas_top20() -> DataFrame:
    """Workload B: rutas de mayor frecuencia."""
    return (
        viajes_filtrados()
        .groupBy("PULocationID", "DOLocationID")
        .agg(
            F.count("*").alias("num_viajes"),
            F.sum("total_amount").alias("ingreso_total"),
        )
        .orderBy(
            F.desc("num_viajes"),
            F.asc("PULocationID"),
            F.asc("DOLocationID"),
        )
        .limit(20)
    )


WORKLOADS: dict[str, Callable[[], DataFrame]] = {
    "zonas_agregadas": consulta_zonas,
    "rutas_top20": consulta_rutas_top20,
}

print("Workloads definidos:", list(WORKLOADS))

Workloads definidos: ['zonas_agregadas', 'rutas_top20']


## 7. Leer los planes físicos de ejecución

Un DataFrame expresa *qué* queremos obtener; el plan físico detalla *cómo* Spark propone ejecutarlo.

Identifica en la salida de `explain(mode="formatted")`:

- `FileScan parquet`: lectura desde Parquet; revisa el listado de columnas leídas (*column pruning*) y filtros empujados cuando corresponda (*predicate pushdown*).
- `Filter` y `Project`: filtrado y selección de columnas.
- `HashAggregate`: agregación parcial/final cuando es aplicable.
- `Exchange`: redistribución, que puede implicar **shuffle**.
- `Sort` y `TakeOrderedAndProject`: ordenamiento, a veces optimizado para los primeros N resultados.
- `AdaptiveSparkPlan`: AQE está habilitado; el plan mostrado **antes de ejecutar** puede no coincidir exactamente con el plan final.

> La presencia de `Exchange` no implica por sí sola un problema. Su costo depende del volumen redistribuido, tamaño de particiones, sesgo de claves y recursos disponibles.

In [6]:
for nombre, construir_consulta in WORKLOADS.items():
    print("\n" + "=" * 80)
    print(f"PLAN FÍSICO INICIAL — {nombre}")
    print("=" * 80)
    construir_consulta().explain(mode="formatted")


PLAN FÍSICO INICIAL — zonas_agregadas
== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [tpep_pickup_datetime#37, trip_distance#40, PULocationID#43, DOLocationID#44, total_amount#52]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/raw/taxi/year=2024/yellow_tripdata_2024-01.parquet, ... 11 entries]
PushedFilters: [IsNotNull(tpep_pickup_datetime), IsNotNull(trip_distance), IsNotNull(total_amount), GreaterThanOrEqual(tpep_pickup_datetime,2024-01-01T00:00), LessThan(tpep_pickup_datetime,2025-01-01T00:00), IsNotNull(PULocationID), IsNotNull(DOLocationID), GreaterThan(trip_distance,0.0), LessThan(trip_distance,100.0), GreaterThanOrEqual(total_amount,0.0)]
ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,trip_distance:double,PULocationID:int,DOLocationID:int,total_amount:double>

(2) Filter
Input 

## 8. Diseño del benchmark

Usaremos `time.perf_counter()` del proceso Python que coordina Spark. Para cada workload:

1. Ejecutamos **una ronda de calentamiento** no contabilizada.
2. Realizamos **cinco repeticiones** cronometradas.
3. Obligamos al cálculo con `collect()` sobre una salida pequeña (zonas agregadas o top 20 rutas), evitando que el resultado completo de millones de viajes viaje al driver.
4. Registramos la duración de cada repetición, mínimo, máximo, mediana y desviación estándar.

Con tiempos $t_1,\dots,t_n$:

$$
\tilde{t}=\operatorname{mediana}(t_1,\ldots,t_n),\qquad
s=\sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(t_i-\bar{t})^2}.
$$

**Interpretación importante:** estas cifras describen el **tiempo total visto por el driver**, incluyendo planificación, ejecución y retorno de resultados pequeños; no son una medición exclusiva del tiempo de CPU de los ejecutores.

**Límites del experimento:**

- Se limpia el **cache administrado por Spark** antes de cada corrida, pero esto **no limpia** el cache del sistema operativo ni el del almacenamiento.
- Como hay calentamiento, nuestros resultados representan esencialmente **ejecuciones con entorno caliente**, no una prueba de arranque en frío.
- La máquina, número de procesos simultáneos, configuración de Spark, volumen de entrada, consultas y repeticiones deben mantenerse constantes para comparar.
- No redondeamos los valores agregados dentro de los workloads: cambiar las transformaciones cambiaría el trabajo medido.
- Las mediciones de distintos workloads no se deben comparar como si fueran la misma consulta.

In [7]:
CALENTAMIENTOS = 1
REPETICIONES = 5


def ejecutar_benchmark(
    nombre: str,
    construir_consulta: Callable[[], DataFrame],
    calentamientos: int = CALENTAMIENTOS,
    repeticiones: int = REPETICIONES,
) -> dict:
    """
    Mide un workload completo: construir el DataFrame y materializarlo.

    `collect()` es razonable aquí porque los dos workloads producen resultados
    reducidos. No reutilices esta función para un DataFrame de millones de filas.
    """
    if calentamientos < 0 or repeticiones < 2:
        raise ValueError("Usa calentamientos >= 0 y repeticiones >= 2.")

    tiempos = []
    filas_esperadas = None

    for indice in range(calentamientos + repeticiones):
        # Invalida únicamente tablas/DataFrames cacheados en Spark.
        # No elimina el cache del sistema operativo.
        spark.catalog.clearCache()

        inicio = time.perf_counter()
        salida = construir_consulta().collect()  # Acción que fuerza la ejecución.
        duracion_s = time.perf_counter() - inicio

        # Una comprobación elemental de consistencia entre repeticiones.
        filas_obtenidas = len(salida)
        if filas_esperadas is None:
            filas_esperadas = filas_obtenidas
        elif filas_obtenidas != filas_esperadas:
            raise AssertionError(
                f"El número de filas cambió en {nombre}: "
                f"{filas_esperadas} -> {filas_obtenidas}"
            )

        es_calentamiento = indice < calentamientos
        etapa = "calentamiento" if es_calentamiento else f"medición {indice - calentamientos + 1}"
        print(f"{nombre:18s} | {etapa:16s} | {duracion_s:8.3f} s | {filas_obtenidas} filas de salida")

        if not es_calentamiento:
            tiempos.append(duracion_s)

    return {
        "workload": nombre,
        "calentamientos": calentamientos,
        "repeticiones": repeticiones,
        "tiempos_s": tiempos,
        "mediana_s": statistics.median(tiempos),
        "media_s": statistics.mean(tiempos),
        "desviacion_s": statistics.stdev(tiempos),
        "minimo_s": min(tiempos),
        "maximo_s": max(tiempos),
        "filas_salida": filas_esperadas,
    }


print(f"Calentamientos: {CALENTAMIENTOS} | Repeticiones por workload: {REPETICIONES}")

Calentamientos: 1 | Repeticiones por workload: 5


## 9. Ejecutar el benchmark de referencia

Este apartado **sí ejecuta Spark de forma intensiva**. Con los parámetros predeterminados se lanzan 12 ejecuciones en total: 1 calentamiento + 5 mediciones para cada uno de los dos workloads.

Antes de ejecutarlo, procura cerrar otras tareas pesadas que puedan competir por CPU, memoria o disco. Si usas Docker Desktop, comprueba que tenga recursos suficientes y evita modificar su asignación durante las corridas.

In [8]:
RESULTADOS_BASELINE = []

for nombre, construir_consulta in WORKLOADS.items():
    print("\n" + "-" * 82)
    RESULTADOS_BASELINE.append(
        ejecutar_benchmark(nombre, construir_consulta)
    )

print("\nBenchmark inicial finalizado.")


----------------------------------------------------------------------------------


zonas_agregadas    | calentamiento    |    3.946 s | 263 filas de salida


zonas_agregadas    | medición 1       |    2.718 s | 263 filas de salida


zonas_agregadas    | medición 2       |    2.305 s | 263 filas de salida


zonas_agregadas    | medición 3       |    2.326 s | 263 filas de salida


...

rutas_top20        | medición 2       |    2.409 s | 20 filas de salida


rutas_top20        | medición 3       |    2.241 s | 20 filas de salida


rutas_top20        | medición 4       |    3.080 s | 20 filas de salida


[Stage 49:===================>                                     (4 + 8) / 12]

rutas_top20        | medición 5       |    2.451 s | 20 filas de salida

Benchmark inicial finalizado.


## 10. Resumen de resultados

La **mediana** será nuestra métrica principal porque es menos sensible a valores extremos que la media. La desviación estándar nos ayuda a interpretar la estabilidad de cada prueba.

No confundas una mejora pequeña con una diferencia concluyente: si la variabilidad entre repeticiones es comparable a la supuesta mejora, conviene aumentar repeticiones o controlar mejor el entorno.

In [9]:
encabezado = (
    f"{'Workload':18s} {'Mediana(s)':>11s} {'Media(s)':>10s} "
    f"{'Desv.(s)':>10s} {'Mín.(s)':>10s} {'Máx.(s)':>10s}"
)
print(encabezado)
print("-" * len(encabezado))

for r in RESULTADOS_BASELINE:
    print(
        f"{r['workload']:18s} {r['mediana_s']:11.3f} "
        f"{r['media_s']:10.3f} {r['desviacion_s']:10.3f} "
        f"{r['minimo_s']:10.3f} {r['maximo_s']:10.3f}"
    )

if not RESULTADOS_BASELINE:
    raise RuntimeError("No hay resultados; ejecuta el apartado 9 antes de continuar.")

Workload            Mediana(s)   Media(s)   Desv.(s)    Mín.(s)    Máx.(s)
--------------------------------------------------------------------------
zonas_agregadas          2.305      2.343      0.226      2.113      2.718
rutas_top20              2.451      2.540      0.319      2.241      3.080


## 11. Inspeccionar Spark UI y el plan después de la acción

La Spark UI permite ver más que el tiempo total:

- **Jobs:** trabajos disparados por cada `collect()`.
- **Stages:** tareas paralelas y sus duraciones.
- **SQL / DataFrame:** plan de consulta ejecutado y métricas por operador.
- **Shuffle Read / Write:** volumen de datos intercambiados.
- **Spill (memory / disk):** escritura temporal asociada a presión de memoria.
- **Task duration:** dispersión de tiempos que puede revelar particiones desbalanceadas.

En la UI, identifica el workload por la actividad reciente y revisa la pestaña **SQL**. Si la URL no abre desde el navegador, revisa los puertos publicados en Docker para Spark UI (normalmente 4040, aunque puede variar).

**AQE:** ejecutar un DataFrame materializado puede permitir inspeccionar su plan adaptativo final; por ello, hacemos una ejecución adicional con fines de diagnóstico. **No se incluye en las mediciones.**

In [10]:
print("Spark UI reportada por la sesión:", spark.sparkContext.uiWebUrl)

# Diagnóstico adicional: no forma parte de RESULTADOS_BASELINE.
df_diagnostico = consulta_rutas_top20()
_ = df_diagnostico.collect()
print("\nPLAN DEL WORKLOAD B TRAS MATERIALIZAR EL DATAFRAME:")
df_diagnostico.explain(mode="formatted")

Spark UI reportada por la sesión: http://455f3b1cac15:4042


[Stage 53:===================>                                     (4 + 8) / 12]


PLAN DEL WORKLOAD B TRAS MATERIALIZAR EL DATAFRAME:
== Physical Plan ==
AdaptiveSparkPlan (18)
+- == Final Plan ==
   ResultQueryStage (11)
   +- TakeOrderedAndProject (10)
      +- * HashAggregate (9)
         +- AQEShuffleRead (8)
            +- ShuffleQueryStage (7), Statistics(sizeInBytes=13.7 MiB, rowCount=3.59E+5)
               +- Exchange (6)
                  +- * HashAggregate (5)
                     +- * Project (4)
                        +- * Filter (3)
                           +- * ColumnarToRow (2)
                              +- Scan parquet  (1)
+- == Initial Plan ==
   TakeOrderedAndProject (17)
   +- HashAggregate (16)
      +- Exchange (15)
         +- HashAggregate (14)
            +- Project (13)
               +- Filter (12)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [tpep_pickup_datetime#615, trip_distance#618, PULocationID#621, DOLocationID#622, total_amount#630]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/ra

El análisis del Stage 29 de Spark UI muestra una distribución relativamente equilibrada del trabajo entre 12 tareas, con duraciones de 4 a 5 segundos y un total de aproximadamente 41.17 millones de registros procesados. No se identificaron indicios relevantes de data skew, presión de memoria o derramamiento a disco. La agregación parcial reduce considerablemente el volumen de información que debe redistribuirse, mientras que Adaptive Query Execution agrupa posteriormente las particiones del shuffle. Estos resultados sugieren que las futuras optimizaciones deben evaluar principalmente la lectura y el procesamiento inicial de datos, así como el efecto del particionamiento, sin asumir que modificar el número de particiones producirá necesariamente mejoras.

### 11.1 — Acceder a Spark UI desde Docker y VS Code

La **Spark UI pertenece a la aplicación Spark**, no a la ejecución individual de una celda. Está disponible **mientras la sesión Spark siga activa**, incluso cuando una consulta haya finalizado. Desaparece si se detiene la aplicación (`spark.stop()`), se cierra el kernel que la sostiene o se apaga el contenedor.

- **Puerto habitual:** `4040`. Si está ocupado, Spark puede utilizar `4041`, `4042`, etc. En la sesión guardada arriba se observa la advertencia de puerto `4040` ocupado y una URL interna terminada en **`:4041`**. En una nueva sesión el puerto puede ser distinto.
- **Nombre del contenedor:** la URL `http://455f3b1cac15:4041` es interna y generalmente no es resoluble desde el navegador de Windows.
- **URL local del navegador:** normalmente `http://localhost:PUERTO` si el puerto está efectivamente **publicado por Docker o redirigido por VS Code**. La celda siguiente calcula el puerto actual y genera un enlace tentativo; **no publica puertos por sí misma**.

#### Opción A — VS Code conectado al contenedor

Si utilizas la extensión **Dev Containers** y VS Code está *adjunto al contenedor*:

1. Ejecuta la celda de diagnóstico inferior con el kernel Spark encendido y anota el puerto obtenido.
2. Abre el panel inferior de VS Code → **Ports (Puertos)**.
3. Selecciona **Forward a Port** e introduce el puerto indicado (por ejemplo, `4041`).
4. Abre la dirección local que muestre VS Code; puede utilizar otro puerto local si el solicitado está ocupado.

> Si VS Code está ejecutándose en Windows y **solo conectaste un kernel Jupyter remoto**, sin estar adjunto al contenedor, el reenvío de puertos de VS Code puede no estar disponible. Utiliza la opción B.

#### Opción B — Docker Desktop / PowerShell (Windows)

Con el contenedor funcionando, verifica si ya tiene el puerto publicado. En PowerShell, **fuera del notebook**, ejecuta:

```powershell
docker ps --format "table {{.Names}}\t{{.Ports}}"
# Reemplaza CONTENEDOR por el nombre o ID que devuelve docker ps
docker port CONTENEDOR 4041/tcp
```

Si el puerto aparece publicado, abre `http://localhost:PUERTO_PUBLICADO` (el puerto del **host**, que puede ser diferente de `4041`). Si `docker port` indica que no existe el mapeo, hay que **recrear el contenedor** conservando los mismos montajes/volúmenes y añadir los flags siguientes **al comando `docker run` que ya utilizas**:

```powershell
-p 127.0.0.1:4040:4040 -p 127.0.0.1:4041:4041
```

Estos flags son **fragmentos del comando `docker run`**, no un comando independiente ni código Python. Hacen accesibles 4040 y 4041 únicamente desde el propio equipo. **Un mapeo no se puede agregar a un contenedor ya creado** mediante `docker run -p`; evita perder datos preservando los montajes y utilizando el mismo directorio del proyecto antes de recrearlo.

Si estás usando **Docker Compose**, agrega `ports` al servicio que ejecuta Spark y recrea ese servicio:

```yaml
ports:
  - "127.0.0.1:4040:4040"
  - "127.0.0.1:4041:4041"
```

#### ¿Qué revisar cuando abras Spark UI?

1. **Jobs:** localizar trabajos exitosos y sus duraciones.
2. **SQL / DataFrame:** identificar la consulta ejecutada y sus operadores físicos (`Exchange`, `HashAggregate`, `AQEShuffleRead`).
3. **Stages:** comparar duración de tareas, tareas lentas, **Shuffle Read / Write** y **Spill (memory/disk)**.
4. **Executors:** inspeccionar uso de memoria, tiempos y tareas por ejecutor. En `local[*]` el entorno no equivale a un clúster distribuido real.

**Importante:** el reporte JSON *no contiene* métricas de shuffle, spill o memoria de Spark UI. No podemos deducirlas solo a partir de `explain()`. Tampoco es necesario volver a ejecutar los benchmarks para abrir los trabajos ya terminados si el **mismo SparkContext** sigue activo.

**Para consultar trabajos cuando la aplicación ya se cerró**, necesitaríamos configurar *Spark Event Logging* antes de iniciar la aplicación y después visualizar los eventos mediante *Spark History Server* (puerto habitual `18080`). No está configurado en este notebook ni puede recuperar retroactivamente una ejecución sin logs.

**Documentación oficial:** [Spark — Monitoring](https://spark.apache.org/docs/latest/monitoring.html) · [Docker — Port publishing](https://docs.docker.com/engine/network/port-publishing/).


In [11]:
# Diagnóstico opcional: ejecutar mientras esta sesión Spark esté activa.
# No dispara acciones costosas sobre el dataset ni modifica los benchmarks.
from urllib.parse import urlparse
import socket

url_ui = spark.sparkContext.uiWebUrl
print("Spark UI (dirección interna reportada por Spark):", url_ui or "deshabilitada")

if url_ui:
    puerto_ui = urlparse(url_ui).port
    if puerto_ui is None:
        print("La URL no indica un puerto válido; revisa la configuración de Spark.")
    else:
        # Esta prueba confirma únicamente que el servicio responde en el contenedor.
        # NO comprueba que Windows pueda abrirlo: para eso se requiere un mapeo/forward.
        try:
            with socket.create_connection(("127.0.0.1", puerto_ui), timeout=2):
                print(f"Servicio Spark UI accesible desde el contenedor: puerto {puerto_ui}")
        except OSError as error:
            print("No se pudo verificar la conexión local dentro del contenedor:", error)

        url_local = f"http://localhost:{puerto_ui}/"
        print("Enlace local tentativo:", url_local)
        print("Solo funciona si Docker publica ese puerto o VS Code lo reenvía.")

        # En un notebook Jupyter se muestra como hipervínculo clicable.
        # Usamos solo el puerto entero calculado a partir de Spark, no HTML arbitrario.
        try:
            from IPython.display import HTML, display
            display(HTML(f'<a href="{url_local}" target="_blank" rel="noopener noreferrer">Abrir Spark UI (si el puerto está expuesto)</a>'))
        except ImportError:
            pass
else:
    print("Verifica spark.ui.enabled y que exista una sesión SparkContext activa.")


Spark UI (dirección interna reportada por Spark): http://455f3b1cac15:4042
Servicio Spark UI accesible desde el contenedor: puerto 4042
Enlace local tentativo: http://localhost:4042/
Solo funciona si Docker publica ese puerto o VS Code lo reenvía.


## 12. Guardar la línea base en un reporte JSON

El reporte permite retomar los números en `05_02`–`05_06`, comprobar que se comparó el mismo entorno y calcular las mejoras porcentuales.

$$
\text{mejora relativa (\%)}=
\frac{t_{\text{baseline}}-t_{\text{optimizado}}}{t_{\text{baseline}}}\times 100.
$$

Ejemplo conceptual: un resultado positivo indica reducción en tiempo, mientras que uno negativo indica regresión. **No calculamos ninguna mejora ahora**, porque todavía no ejecutamos una versión optimizada.

Se escribe solamente un JSON pequeño dentro de `docs/benchmarks/`; **no se sobrescriben ni modifican los datos Parquet**.

In [12]:
DIRECTORIO_REPORTES = PROJECT_ROOT / "docs" / "benchmarks"
DIRECTORIO_REPORTES.mkdir(parents=True, exist_ok=True)

ETIQUETA_PERIODO = MESES[0] if len(MESES) == 1 else f"{MESES[0]}_a_{MESES[-1]}"
RUTA_REPORTE = DIRECTORIO_REPORTES / f"05_01_baseline_{ETIQUETA_PERIODO}.json"

REPORTE = {
    "notebook": "05_01_diagnostico_benchmarking",
    "fase": "baseline",
    "creado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "periodo": {
        "meses": MESES,
        "inicio_inclusivo": FECHA_INICIO.isoformat(),
        "fin_exclusivo": FECHA_FIN_EXCLUSIVA.isoformat(),
        "archivos": ARCHIVOS_RELATIVOS,
        "tamano_archivos_mib": round(TAMANO_ENTRADA_MB, 3),
    },
    "ambiente": ENTORNO,
    "filtros": {
        "fecha": "fecha_inicio <= tpep_pickup_datetime < fecha_fin_exclusiva",
        "PULocationID": "no nulo",
        "DOLocationID": "no nulo",
        "trip_distance": "0 < distancia < 100",
        "total_amount": "monto >= 0",
    },
    "condiciones": {
        "cache_spark": "clearCache antes de cada ejecución",
        "cache_sistema_operativo": "no controlado",
        "accion": "collect sobre resultados agregados pequeños",
        "cronometro": "time.perf_counter (Python driver)",
    },
    "resultados": RESULTADOS_BASELINE,
}

if RUTA_REPORTE.exists():
    print("Ya existe el baseline; se conserva sin sobrescribir:", ruta_publicable(RUTA_REPORTE))
    print("Las mediciones de esta sesión están en RESULTADOS_BASELINE, pero no reemplazan el JSON previo.")
else:
    with RUTA_REPORTE.open("x", encoding="utf-8") as archivo:
        json.dump(REPORTE, archivo, ensure_ascii=False, indent=2)
    print("Nuevo baseline guardado en:", ruta_publicable(RUTA_REPORTE))

Ya existe el baseline; se conserva sin sobrescribir: docs/benchmarks/05_01_baseline_2024-01_a_2024-12.json
Las mediciones de esta sesión están en RESULTADOS_BASELINE, pero no reemplazan el JSON previo.


## 13. Comparación enero vs. año completo

Compararemos los JSON registrados por **este mismo notebook**, con las mismas consultas y configuraciones.

- **Factor de tamaño:** tamaño físico de los Parquet anuales dividido entre el de enero. Es solo una referencia del volumen de archivos, no de la cantidad de datos efectivamente leída después del *column pruning*.
- **Factor de tiempo:** mediana anual dividida entre la mediana mensual. Un factor de 4 significa que el experimento anual tardó cuatro veces lo que el de enero.
- **Coeficiente de variación (CV):** desviación estándar / media. Ayuda a evaluar estabilidad, pero con cinco repeticiones no es una medida concluyente.

**Precaución:** no confundas la diferencia de tiempos enero vs. año con una ganancia de optimización. El aumento de datos y el comportamiento de las cachés del sistema operativo hacen que no exista obligación de escalar proporcionalmente.

Esta celda **solo lee JSON existentes** y no ejecuta de nuevo los workloads. Si falta el JSON de enero, deja su archivo bajo `docs/benchmarks/` para habilitar la comparación.

In [13]:
from pathlib import Path

RUTA_ENERO = DIRECTORIO_REPORTES / "05_01_baseline_2024-01.json"
RUTA_ANUAL = DIRECTORIO_REPORTES / "05_01_baseline_2024-01_a_2024-12.json"


def leer_reporte(ruta: Path) -> dict:
    """Carga un informe existente sin lanzar nuevas acciones de Spark."""
    with ruta.open("r", encoding="utf-8") as archivo:
        return json.load(archivo)


if RUTA_ENERO.is_file() and RUTA_ANUAL.is_file():
    reporte_enero = leer_reporte(RUTA_ENERO)
    reporte_anual = leer_reporte(RUTA_ANUAL)

    trabajos_enero = {r["workload"]: r for r in reporte_enero["resultados"]}
    trabajos_anual = {r["workload"]: r for r in reporte_anual["resultados"]}
    trabajos_comunes = sorted(trabajos_enero.keys() & trabajos_anual.keys())

    factor_archivos = (
        reporte_anual["periodo"]["tamano_archivos_mib"]
        / reporte_enero["periodo"]["tamano_archivos_mib"]
    )
    print(f"Factor de tamaño físico de los Parquet: {factor_archivos:.2f}×")
    print(f"{'Workload':18s} {'Enero (s)':>11s} {'Anual (s)':>11s} "
          f"{'Tiempo ×':>10s} {'CV anual':>10s}")
    print("-" * 66)

    for nombre in trabajos_comunes:
        enero, anual = trabajos_enero[nombre], trabajos_anual[nombre]
        factor_tiempo = anual["mediana_s"] / enero["mediana_s"]
        cv_anual = anual["desviacion_s"] / anual["media_s"]
        print(f"{nombre:18s} {enero['mediana_s']:11.3f} "
              f"{anual['mediana_s']:11.3f} {factor_tiempo:10.2f}× {cv_anual:10.1%}")
else:
    print("Comparación no disponible. Faltan los siguientes reportes:")
    for ruta in (RUTA_ENERO, RUTA_ANUAL):
        if not ruta.is_file():
            print(" -", ruta_publicable(ruta))


Factor de tamaño físico de los Parquet: 13.87×
Workload             Enero (s)   Anual (s)   Tiempo ×   CV anual
------------------------------------------------------------------
rutas_top20              0.574       2.296       4.00×      13.0%
zonas_agregadas          0.578       2.036       3.52×      13.9%


## 14. Referencia de resultados: validar corrección además de velocidad

Una optimización no es válida si acelera una consulta pero **cambia su resultado**. El JSON del benchmark actual solo comprueba cuántas filas devuelve cada workload (263 zonas y 20 rutas en la medición anual); esto no demuestra equivalencia de los indicadores.

Crearemos un **segundo archivo JSON**, independiente del reporte de tiempos, con el resultado de las dos consultas originales:

- Para `zonas_agregadas`, ordenamos las zonas y registramos `num_viajes`, `distancia_media` e `importe_medio`.
- Para `rutas_top20`, registramos las 20 rutas y sus indicadores, ordenadas por identificadores para serialización estable.
- Los conteos y las claves se deben comparar **exactamente**; los importes y promedios admiten una tolerancia pequeña, ya que los cambios de particionamiento pueden alterar el orden de sumas de punto flotante.

**Esta es una prueba funcional adicional, fuera de las mediciones**: ejecuta Spark una vez por workload, pero **no altera las medianas guardadas**. No deberíamos sobrescribir esta referencia con resultados optimizados.

In [14]:
import math

CLAVES_WORKLOAD = {
    "zonas_agregadas": ("PULocationID",),
    "rutas_top20": ("PULocationID", "DOLocationID"),
}


def salida_ordenada(nombre: str, df: DataFrame) -> list[dict]:
    """Recolecta solo las salidas pequeñas y ordena las filas por su clave."""
    claves = CLAVES_WORKLOAD[nombre]
    filas = [fila.asDict(recursive=True) for fila in df.collect()]
    return sorted(filas, key=lambda fila: tuple(fila[c] for c in claves))


REFERENCIA_LOGICA = {
    "notebook": "05_01_diagnostico_benchmarking",
    "periodo": list(MESES),
    "filtros": dict(REPORTE["filtros"]),
    "resultados": {
        nombre: salida_ordenada(nombre, construir_consulta())
        for nombre, construir_consulta in WORKLOADS.items()
    },
}

# Derivamos viajes admitidos de las agregaciones, sin agregar otro count() de gran volumen.
viajes_admitidos = sum(
    fila["num_viajes"]
    for fila in REFERENCIA_LOGICA["resultados"]["zonas_agregadas"]
)
REFERENCIA_LOGICA["viajes_admitidos_tras_filtros"] = viajes_admitidos

RUTA_REFERENCIA = (
    DIRECTORIO_REPORTES / f"05_01_referencia_logica_{ETIQUETA_PERIODO}.json"
)
if RUTA_REFERENCIA.exists():
    # No reemplazar inadvertidamente la fuente de verdad del experimento original.
    referencia_guardada = leer_reporte(RUTA_REFERENCIA)
    print("Referencia preexistente: se comparará en lugar de sobrescribirse.")
else:
    with RUTA_REFERENCIA.open("x", encoding="utf-8") as archivo:
        json.dump(REFERENCIA_LOGICA, archivo, ensure_ascii=False, indent=2, allow_nan=False)
    referencia_guardada = REFERENCIA_LOGICA
    print("Nueva referencia guardada:", ruta_publicable(RUTA_REFERENCIA))


def comprobar_equivalencia(referencia: dict, candidato: dict,
                           tolerancia_rel: float = 1e-8,
                           tolerancia_abs: float = 1e-5) -> None:
    """Valida mismas claves, conteos exactos y medidas numéricas casi iguales."""
    if referencia["periodo"] != candidato["periodo"]:
        raise AssertionError("Los períodos de las consultas son distintos.")
    if referencia["filtros"] != candidato["filtros"]:
        raise AssertionError("Los filtros de las consultas son distintos.")

    for nombre, filas_originales in referencia["resultados"].items():
        claves = CLAVES_WORKLOAD[nombre]
        filas_candidatas = candidato["resultados"][nombre]

        def indexar(filas):
            tabla = {}
            for fila in filas:
                clave = tuple(fila[c] for c in claves)
                if clave in tabla:
                    raise AssertionError(f"Clave duplicada en {nombre}: {clave}")
                tabla[clave] = fila
            return tabla

        originales = indexar(filas_originales)
        candidatos = indexar(filas_candidatas)
        if originales.keys() != candidatos.keys():
            raise AssertionError(f"Cambió el conjunto de claves en {nombre}.")

        for clave, fila_original in originales.items():
            fila_candidata = candidatos[clave]
            if fila_original.keys() != fila_candidata.keys():
                raise AssertionError(f"Cambió el esquema de resultados: {nombre} {clave}")
            for columna, esperado in fila_original.items():
                observado = fila_candidata[columna]
                if isinstance(esperado, float) and isinstance(observado, (int, float)):
                    iguales = math.isclose(
                        esperado, observado,
                        rel_tol=tolerancia_rel,
                        abs_tol=tolerancia_abs,
                    )
                else:
                    iguales = (esperado == observado)
                if not iguales:
                    raise AssertionError(
                        f"Diferencia en {nombre} {clave} / {columna}: "
                        f"original={esperado}, candidato={observado}"
                    )
    print("Equivalencia funcional verificada: claves, conteos y agregados.")


comprobar_equivalencia(referencia_guardada, REFERENCIA_LOGICA)
print("Viajes admitidos tras filtros:", f"{viajes_admitidos:,}")
print("Filas por workload:", {
    nombre: len(filas)
    for nombre, filas in REFERENCIA_LOGICA["resultados"].items()
})


[Stage 61:===================>                                     (4 + 8) / 12]

Referencia preexistente: se comparará en lugar de sobrescribirse.
Equivalencia funcional verificada: claves, conteos y agregados.
Viajes admitidos tras filtros: 39,833,077
Filas por workload: {'zonas_agregadas': 263, 'rutas_top20': 20}


## 15. Entorno Docker: límites reales y lectura de la Spark UI

El reporte muestra `spark.driver.memory: no especificada`. Eso **no significa memoria ilimitada**: indica que no se fijó explícitamente esa propiedad. Docker y el kernel Linux pueden imponer límites de CPU o RAM que conviene conservar al comparar optimizaciones.

En Linux, los límites del contenedor pueden consultarse en los archivos de **cgroups**. El diagnóstico siguiente es opcional y no forma parte del tiempo medido. Si el sistema no expone esos archivos, se informará sin fallar el notebook.

Respecto a los planes de ejecución ya observados:

- `Exchange` es evidencia de redistribución de datos; no demuestra por sí mismo el volumen exacto de shuffle.
- `HashAggregate` parcial y final explica la estrategia de agregación.
- `TakeOrderedAndProject` implementa la selección ordenada de las 20 rutas.
- `AQEShuffleRead` en el plan final demuestra que AQE está interviniendo; **200 particiones configuradas no necesariamente equivalen a 200 particiones efectivas finales**.

Para identificar cuellos de botella concretos, complementa los tiempos del JSON con **Shuffle Read / Write, Spill y tiempos de tareas** en Spark UI. No se deben inventar esas métricas a partir del plan físico.

In [15]:
def leer_archivo_cgroup(ruta: str) -> str | None:
    """Lee un límite cgroup si existe en el kernel del contenedor Linux."""
    archivo = Path(ruta)
    try:
        return archivo.read_text(encoding="utf-8").strip()
    except (FileNotFoundError, PermissionError, OSError):
        return None


DIAGNOSTICO_CGROUPS = {
    "cgroup_v2_memory_max_bytes": leer_archivo_cgroup("/sys/fs/cgroup/memory.max"),
    "cgroup_v2_cpu_max": leer_archivo_cgroup("/sys/fs/cgroup/cpu.max"),
    "cgroup_v1_memory_limit_bytes": leer_archivo_cgroup(
        "/sys/fs/cgroup/memory/memory.limit_in_bytes"
    ),
    "cgroup_v1_cpu_quota_us": leer_archivo_cgroup(
        "/sys/fs/cgroup/cpu/cpu.cfs_quota_us"
    ),
}

for concepto, valor in DIAGNOSTICO_CGROUPS.items():
    print(f"{concepto:34s}: {valor or 'no disponible'}")

print("\nInterpretación: 'max' significa sin límite cgroup v2 para ese recurso;")
print("valores específicos son límites potenciales del contenedor, no métricas de uso.")


cgroup_v2_memory_max_bytes        : max
cgroup_v2_cpu_max                 : max 100000
cgroup_v1_memory_limit_bytes      : no disponible
cgroup_v1_cpu_quota_us            : no disponible

Interpretación: 'max' significa sin límite cgroup v2 para ese recurso;
valores específicos son límites potenciales del contenedor, no métricas de uso.


## 16. Validación final y conclusiones para documentar

### Lista de comprobación

- [x] El kernel corresponde al entorno Python del contenedor y Spark reconoce la sesión (Linux, Python 3.11.17, Spark 4.1.3).
- [x] Se localizaron los doce archivos Parquet sin errores de esquema o rutas (660.898 MiB).
- [x] Se obtuvieron planes físicos de ambos workloads.
- [x] Se completaron cinco mediciones de cada consulta, con una ejecución de calentamiento.
- [x] Se conservó el JSON anual original de referencia y se comparó con enero.
- [x] Se generó una referencia lógica de resultados y se verificó equivalencia para una segunda ejecución de las consultas originales.
- [x] Se consultaron los límites cgroups disponibles del contenedor.
- [ ] **Pendiente de inspección manual:** abrir Spark UI y registrar métricas reales de shuffle, spill y tiempos de tareas.

### Respuestas a las preguntas de interpretación

#### 1. ¿Qué operadores físicos identificamos en cada workload? ¿Dónde aparecen los `Exchange`?

Ambas consultas parten de la lectura columnar **`Scan parquet`**, luego aplican **`Filter`** y **`Project`**; el plan ejecutado puede incluir **`ColumnarToRow`**. La agregación utiliza dos niveles de **`HashAggregate`** (parcial y final), separados por **`Exchange`**, que redistribuye resultados parciales de acuerdo con las claves de agrupamiento.

- **Agregación por zonas:** `Exchange hashpartitioning(PULocationID, 200)` entre las agregaciones parciales y finales.
- **Top 20 rutas:** `Exchange hashpartitioning(PULocationID, DOLocationID, 200)`; además aparece **`TakeOrderedAndProject`** por `orderBy(...).limit(20)`.
- En el **plan final** del workload de rutas figura **`AQEShuffleRead`** con particiones *coalesced*, evidencia de que **Adaptive Query Execution (AQE)** ajustó la lectura del shuffle. Los **200** son una configuración nominal: las particiones efectivamente leídas pueden cambiar con AQE.
- No se observó un operador `Sort` separado en el plan inicial de las rutas; la selección ordenada se resuelve mediante `TakeOrderedAndProject`.

**Interpretación:** `Exchange` marca un punto de movimiento de datos, pero su presencia no permite cuantificar los bytes transferidos ni afirmar por sí sola que exista un cuello de botella. Para eso consultaremos métricas de **Stages** en Spark UI.

#### 2. ¿Qué workload presenta mayor mediana y qué trabajo adicional puede explicar la diferencia?

| Workload | Mediana anual guardada (baseline) | Mediana anual de la nueva ejecución |
|:--|--:|--:|
| Zonas agregadas | **2.036 s** | **2.137 s** |
| Top 20 rutas | **2.296 s** | **2.319 s** |

En ambas ejecuciones **Top 20 rutas** presenta mayor mediana. Para el baseline anual original, la diferencia es de **0.260 s**, aproximadamente **12.8 %** sobre la mediana de zonas. Es compatible con una **cardinalidad mayor de claves** al agrupar por `(PULocationID, DOLocationID)` y el trabajo adicional de ordenar y seleccionar los 20 pares más frecuentes.

El plan adaptativo de rutas incluye estadísticas de una etapa de shuffle de aproximadamente **3.59 × 10⁵ filas** y **13.7 MiB**; son estadísticas de esa etapa, **no** una medición exhaustiva de tráfico de shuffle ni una prueba causal del mayor tiempo. La Spark UI nos permitirá determinar dónde se concentra el costo.

#### 3. ¿Las repeticiones presentan tiempos similares? ¿Qué pudo contribuir a las variaciones?

En la **última ejecución** contenida en este notebook:

| Medida | Zonas agregadas | Top 20 rutas |
|:--|--:|--:|
| Media | 2.197 s | 2.308 s |
| Mediana | 2.137 s | 2.319 s |
| Desviación estándar | 0.165 s | 0.118 s |
| Coeficiente de variación (desviación / media) | **7.5 %** | **5.1 %** |

Los tiempos son **razonablemente estables para una línea base exploratoria**, aunque no idénticos. Entre las causas plausibles de variación están la caché de archivos del sistema operativo, la planificación de tareas, el trabajo de otros procesos, la compilación/JIT y el comportamiento adaptativo de Spark.

En el **JSON anual original**, los CV fueron algo mayores (**13.9 %** zonas y **13.0 %** rutas), por lo que no conviene concluir que diferencias muy pequeñas entre configuraciones representen mejoras. Con cinco mediciones podemos comparar magnitudes, pero no garantizar significancia estadística de cambios marginales.

#### 4. ¿Qué debemos mantener fijo para comparar el baseline con una optimización?

- **Entrada y lógica:** mismos doce archivos de 2024, intervalo de fechas, filtros de calidad, columnas seleccionadas, agregaciones y ordenamiento final.
- **Condiciones de medición:** una ejecución de calentamiento, cinco repeticiones, `collect()` de las salidas pequeñas, cronómetro con `time.perf_counter()` y mediana como medida principal.
- **Entorno:** misma versión de Spark, imagen Docker, modo `local[*]`, recursos disponibles, versión de Java/Python y condiciones de carga del sistema en la medida de lo posible.
- **Configuración de Spark:** conservar AQE y las demás opciones **excepto la variable de optimización estudiada**; por ejemplo, para experimentar con `spark.sql.shuffle.partitions`, esa será la variable controlada que sí cambiaremos y registraremos.
- **Corrección:** la versión optimizada tiene que superar `comprobar_equivalencia()` frente a la **referencia lógica original**. El notebook reportó **39,833,077 viajes admitidos** después de los filtros, **263** zonas y **20** rutas en el resultado final.

Las comparaciones **enero vs. año completo** miden la escala de procesamiento, no una optimización. Los Parquet físicos pasaron de **47.647 MiB** a **660.898 MiB** (**13.87×**), mientras las medianas pasaron de **0.578 a 2.036 s** para zonas (**3.52×**) y de **0.574 a 2.296 s** para rutas (**4.00×**). Esto no demuestra por sí mismo escalabilidad lineal.

#### 5. ¿Cuál es la limitación de medir con el sistema operativo caliente frente a una prueba en frío?

El benchmark ejecuta `spark.catalog.clearCache()` antes de cada repetición, **pero esta instrucción solo afecta cachés administradas por Spark**: no borra la **page cache del sistema operativo**. Tras una primera lectura, parte de los Parquet puede permanecer en la memoria del sistema y las siguientes lecturas resultar más rápidas.

Por ello, nuestros tiempos representan principalmente **ejecuciones repetidas en un entorno potencialmente caliente**, no necesariamente la latencia del primer procesamiento desde disco. Es una línea base útil **si aplicamos el mismo protocolo** a las versiones optimizadas, pero los resultados no deben generalizarse a ejecuciones frías, a otros equipos o a un clúster distribuido sin una evaluación adicional.

### Conclusión técnica

Disponemos de una línea base **anual reproducible**, dos consultas fijas y una referencia para comprobar que las optimizaciones conservan los resultados. La siguiente fase debe medir el efecto de cambios en las particiones y, cuando esté accesible la Spark UI, registrar métricas reales de **shuffle**, **spill** y **tiempo de tareas** para fundamentar los diagnósticos.

### Próximo notebook: `05_02 — Particionamiento y shuffle`

Analizaremos particiones de entrada y de shuffle, `repartition()` frente a `coalesce()`, `spark.sql.shuffle.partitions` y el efecto de AQE. Compararemos **las mismas consultas** con la línea base antes de aprobar cualquier optimización.

> Mantén activo el kernel de Jupyter y evita llamar a `spark.stop()` mientras quieras consultar Spark UI. **No hace falta repetir los benchmarks existentes solo para ejecutar la nueva celda de acceso**, siempre que la sesión todavía esté disponible.
